In [ ]:
# Cell 1: Convert QUBO Matrix to Ising Hamiltonian & Run QAOA
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

Q = np.load("qubo_matrix.npy")
num_qubits = Q.shape[0]

def qubo_to_ising(Q):
    pauli_list = []
    for i in range(num_qubits):
        for j in range(num_qubits):
            if i == j:
                coeff = Q[i, i] / 2.0
                z_string = ["I"] * num_qubits
                z_string[i] = "Z"
                pauli_list.append(("".join(z_string), -coeff))
            elif i < j:
                coeff = Q[i, j] / 4.0
                z_string_ij = ["I"] * num_qubits
                z_string_ij[i] = "Z"
                z_string_ij[j] = "Z"
                pauli_list.append(("".join(z_string_ij), coeff))
    return SparsePauliOp.from_list(pauli_list)

cost_hamiltonian = qubo_to_ising(Q)
backend = AerSimulator()
qaoa_ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=1)

def evaluate_energy(parameters):
    bound_circuit = qaoa_ansatz.assign_parameters(parameters)
    bound_circuit.measure_all()
    result = backend.run(bound_circuit, shots=1024).result()
    counts = result.get_counts()
    
    total_energy = 0.0
    for bitstring, count in counts.items():
        x = np.array([int(b) for b in bitstring[::-1]])
        total_energy += (x.T @ Q @ x) * count
    return total_energy / 1024.0

res = minimize(evaluate_energy, [0.1, 0.1], method='COBYLA', options={'maxiter': 50})
print("\nQAOA Optimization Completed!")
print(f"Optimal Variational Parameters: {res.x}")
print(f"Minimized Cost: {res.fun:,.0f} IRR")

np.save("optimal_params.npy", res.x)